In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr
from dotenv import load_dotenv
from earth2studio.data import GFS, prep_data_array
from earth2studio.models.dx import CorrDiffTaiwan
from earth2studio.utils.coords import map_coords
from earth2studio.utils.time import to_time_array
from matplotlib.animation import FuncAnimation, PillowWriter

Load the data used for training (x and y, but only a small chunk)

In [ ]:
datapath = Path.home() / "ml-ds_data" / "cwa_dataset_storm.zarr"
ds = xr.open_zarr(datapath, consolidated=False)

In [ ]:
time_h = ds.time.values.astype("datetime64[h]")
time_mask = (
    (time_h >= np.datetime64("2021-09-10", "h"))
    & (time_h <= np.datetime64("2021-09-13", "h"))
    & ((time_h.astype("int64") % 6) == 0)
 )
ds = ds.isel(time=time_mask)

In [ ]:
def make_gif(da):
    da = da.transpose("time", "south_north", "west_east")
    vmin = float(da.min().values)
    vmax = float(da.max().values)

    fig, ax = plt.subplots(figsize=(6, 5), dpi=120)
    im = ax.imshow(
        da.isel(time=0).values,
        origin="lower",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        animated=True,
        aspect="auto",
    )
    fig.colorbar(im, ax=ax, label=da.name or "value")
    title = ax.set_title("")
    ax.set_xlabel("west_east")
    ax.set_ylabel("south_north")

    def _fmt_time(t):
        try:
            return np.datetime_as_string(t, unit="m")
        except Exception:
            return str(t)

    def _update(i):
        im.set_data(da.isel(time=i).values)
        title.set_text(f"time = {_fmt_time(da.time.values[i])}")
        return im, title

    anim = FuncAnimation(fig, _update, frames=da.sizes["time"], interval=200, blit=False)

    out_dir = Path("outputs")
    out_dir.mkdir(exist_ok=True)
    gif_path = out_dir / f"{da.name}.gif"
    anim.save(gif_path, writer=PillowWriter(fps=5))
    plt.close(fig)

Make video from the training target data

In [ ]:
da_temp = ds.cwb[:, 1, :, :]
da_temp.name = "temperature"
da_u = ds.cwb[:, 2, :, :]
da_v = ds.cwb[:, 3, :, :]
da_speed = np.sqrt(da_u**2 + da_v**2)
da_speed.name = "speed"

In [ ]:
for da in [da_temp, da_speed]:
    make_gif(da)

Make predictions

In [ ]:
load_dotenv()

# Create CorrDiff model
package = CorrDiffTaiwan.load_default_package()
corrdiff = CorrDiffTaiwan.load_model(package)

# Create the data source
data = GFS()

In [ ]:
time, number_of_samples = ds.time.values, 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
corrdiff = corrdiff.to(device)
corrdiff.number_of_samples = number_of_samples

In [ ]:
pred_time = to_time_array(ds.time.values.astype("datetime64[h]"))

In [ ]:
%%capture
fetched_data = data(pred_time, corrdiff.input_coords()["variable"])

In [ ]:
x, coords = prep_data_array(fetched_data, device=device) # type: ignore
x, coords = map_coords(x, coords, corrdiff.input_coords())
y_pred, coords_out = corrdiff(x, coords)

In [ ]:
pred_np = y_pred.detach().cpu().numpy()
dims = list(coords_out.keys())

da_pred = xr.DataArray(pred_np, dims=dims, name="corrdiff_pred")

# Attach coords from coords_out, including 2D lat/lon grids.
for key, value in coords_out.items():
    arr = np.asarray(value)

    if arr.ndim == 1 and key in da_pred.dims and arr.shape[0] == da_pred.sizes[key]:
        da_pred = da_pred.assign_coords({key: arr})
        continue

    if arr.ndim == 2:
        assigned = False
        for i, d1 in enumerate(da_pred.dims):
            for d2 in da_pred.dims[i + 1 :]:
                shape_2d = (da_pred.sizes[d1], da_pred.sizes[d2])
                if arr.shape == shape_2d:
                    da_pred = da_pred.assign_coords({key: ((d1, d2), arr)})
                    assigned = True
                    break
                if arr.shape == (shape_2d[1], shape_2d[0]):
                    da_pred = da_pred.assign_coords({key: ((d1, d2), arr.T)})
                    assigned = True
                    break
            if assigned:
                break

# Keep one sample/member if prediction has an ensemble-like axis.
for maybe_member_dim in ["sample", "member", "ensemble"]:
    if maybe_member_dim in da_pred.dims:
        da_pred = da_pred.isel({maybe_member_dim: 0}, drop=True)

var_dim = None
for candidate in ["variable", "channel", "var"]:
    if candidate in da_pred.dims:
        var_dim = candidate
        break

if var_dim is None:
    raise ValueError(f"Could not find variable dimension in dims={da_pred.dims}")

temperature_pred = da_pred.sel({var_dim: "t2m"})
u10m_pred = da_pred.sel({var_dim: "u10m"})
v10m_pred = da_pred.sel({var_dim: "v10m"})
wind_speed_pred = np.sqrt(u10m_pred**2 + v10m_pred**2)

temperature_pred = temperature_pred.squeeze(drop=True)
wind_speed_pred = wind_speed_pred.squeeze(drop=True)

temperature_pred.name = "temperature_pred"
wind_speed_pred.name = "wind_speed_pred"

rename_map = {}
for dim in temperature_pred.dims:
    if dim in ["batch", "step", "lead_time"]:
        rename_map[dim] = "time"
    elif dim in ["lat", "latitude", "y", "south_north"]:
        rename_map[dim] = "south_north"
    elif dim in ["lon", "longitude", "x", "west_east"]:
        rename_map[dim] = "west_east"

temperature_pred = temperature_pred.rename(rename_map)
wind_speed_pred = wind_speed_pred.rename(rename_map)

# Ensure dim order expected by make_gif.
temperature_pred = temperature_pred.transpose("time", "south_north", "west_east")
wind_speed_pred = wind_speed_pred.transpose("time", "south_north", "west_east")

In [ ]:
for da in [temperature_pred, wind_speed_pred]:
    make_gif(da)